# 01 · Load and view the MaleCNS connectome

**Goal:** open the real fruit-fly data, and get a feel for what's in it before we build anything from it.

**MaleCNS v1.0** is the wiring diagram of an adult male *Drosophila* central nervous system: ~211,000 neurons and ~150 million synapse connections, produced by
the FlyEM project at HHMI Janelia Research Campus in collaboration with Google. It is released under **CC BY 4.0**: if you use it, credit them.

You will learn
- the three files and what each holds
- how the repo's `Connectome` class opens them (`src/fly_mario_fly/fly/connectome.py`)
- neuron **types**, **sides**, **soma positions**, **transmitters**
- why the 1 GB synapse file is **streamed**, never loaded whole

> **Everything in this notebook is teal** (real data), in the repo's honesty rule: *teal = real wiring, orange = invented by us*.

## 0. Point the notebook at your data

The notebook never downloads anything. Follow `tutorials/README.md` ("Getting the data"), then set the folder below.
It is the folder that **contains** `malecns_v1.0/`.

In [ ]:
DATA_DIR = "~/fly_mario_fly_data"      # <-- change this to the folder that holds malecns_v1.0/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import _helpers as H
H.use_data_folder(DATA_DIR)            # checks the 3 files exist and tells the repo where they are (no download)

## 1. Open it

`Connectome()` reads the small annotation file into memory (about 7 seconds) and keeps a path to the big weights file.
It also checks that the data is the version the repo was built for: the neuron count and the count of the types we use must match (`N_BODIES`, `groups.yaml`).

In [ ]:
from fly_mario_fly.fly.connectome import Connectome, FILES

conn = Connectome()
print("neurons (bodies):", len(conn.ids))
for key, name in FILES.items():
    p = conn.paths[key]
    print(f"{key:18s} {p.name:60s} {p.stat().st_size / 1e6:9.1f} MB")

## 2. What does a neuron look like here?

Per neuron (a "body" with an id) the annotation table gives: a **type** (like `LC4`, a cell type defined by the FlyEM team), a **side**, a **subclass**, and where its cell body (**soma**) is.
`conn` keeps them as parallel numpy arrays sorted by body id.

In [ ]:
ann = pd.DataFrame({"bodyId": conn.ids, "type": conn.type, "side": conn.side, "subclass": conn.subclass})
ann.sample(8, random_state=0)

In [ ]:
print("Neurons with no type name yet:", f"{(conn.type == '').mean():.0%}")
types = pd.Series(conn.type[conn.type != ""]).value_counts()
print("distinct named types:", len(types))
types.head(12)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
types.head(15)[::-1].plot.barh(ax=axes[0], color=H.TEAL); axes[0].set_title("most common named types")
pd.Series(conn.side).value_counts().plot.bar(ax=axes[1], color=H.TEAL); axes[1].set_title("side (L, R, M middle, ? unknown)")
plt.tight_layout(); plt.show()

Many types are visual neurons (`R1-R6` are photoreceptors; `Tm3`, `T3`... are optic-lobe cells). The repo only needs a few cell types — the ones that *do something for Mario*.
They're listed in `src/fly_mario_fly/fly/groups.yaml`: for example **LC4** and **LPLC2** (looming detectors), **DNp01** (the Giant Fiber), **TTMn** (a jump muscle motor neuron).

In [ ]:
from fly_mario_fly.fly.groups import type_counts
for t, expected in type_counts().items():
    print(f"{t:6s} expected {expected:4d}   in the data {int((conn.type == t).sum()):4d}")

Those counts are *checked at load time* (that is what `_check()` does): if Janelia ever changed the data underneath us, the repo would fail loudly instead of silently giving different results.

## 3. Where are the cell bodies?

`conn.soma_positions(ids)` gives each neuron's soma location in **voxel** coordinates (the EM images have 8 nm voxels). About a third of the neurons have no soma annotation (`nan`).

In [ ]:
has = ~np.isnan(conn.soma_xyz[:, 0])
print(f"{has.mean():.0%} of neurons have a soma position")

cast = {"LC4": "tab:blue", "LPLC2": "tab:green", "DNp01": "tab:red", "TTMn": "gold", "DNp09": "tab:cyan"}
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (i, j, lab) in zip(axes, [(0, 1, "x vs y"), (0, 2, "x vs z")]):
    idx = np.flatnonzero(has)
    idx = np.random.default_rng(0).choice(idx, 15000, replace=False)
    ax.scatter(conn.soma_xyz[idx, i], conn.soma_xyz[idx, j], s=1, c="lightgrey")
    for t, c in cast.items():
        ids = conn.body_ids(types=[t])
        p = conn.soma_positions(ids)
        ax.scatter(p[:, i], p[:, j], s=12, c=c, label=f"{t} ({len(ids)})")
    ax.set_title(f"soma positions, {lab} (voxels)"); ax.set_aspect("equal"); ax.invert_yaxis()
axes[0].legend(markerscale=2, fontsize=8)
plt.show()

Grey: a random sample of all neurons that have a soma position. The coloured ones are the cast we use. The looming detectors (LC4, LPLC2) have their cell bodies on the left and right sides of the brain, next to the optic lobes. The two Giant Fibers (DNp01, red) sit near the middle of the brain, and the two jump motor neurons (TTMn, yellow) are far below, in the **ventral nerve cord** (the fly's "spinal cord"): see the x vs z panel.

`conn.body_ids(types=[...], prefix=None, side=None)` is how the rest of the repo picks neurons: by type (or type prefix) and optionally side.

In [ ]:
for side in ("L", "R"):
    print(f"LC4 side {side}:", len(conn.body_ids(types=["LC4"], side=side)))
print("all types starting with 'JO' (hearing):", len(conn.body_ids(prefix="JO")))
print("all types starting with 'ORN' (smell)  :", len(conn.body_ids(prefix="ORN")))

## 4. Excitatory or inhibitory? The transmitter

Each neuron has a consensus neurotransmitter. The repo applies **Dale's law**: one sign per neuron. Acetylcholine (and dopamine, as the repo treats it) = +1, GABA and glutamate = −1, unknown = 0.
(Glutamate can be either in real flies; treating it as inhibitory is a simplification made by the repo. Look at `NT_SIGN` in `connectome.py`. That is why the TTMn motor neurons below show −1: they are glutamatergic. Nothing downstream reads their output as an input, so it doesn't matter for the fly's behaviour.)

In [ ]:
from fly_mario_fly.fly.connectome import NT_SIGN
print(NT_SIGN)
for t in ["LC4", "LPLC2", "DNp01", "TTMn", "DNp09"]:
    s = conn.signs(conn.body_ids(types=[t]))
    print(f"{t:6s} +1: {(s > 0).sum():4d}   -1: {(s < 0).sum():3d}   unknown: {(s == 0).sum():3d}")

## 5. The synapse file: 150 million rows, never loaded whole

`connectome-weights-...feather` is a table `(body_pre, body_post, weight)`: how many synapses neuron *pre* makes onto neuron *post*. It's about 1 GB on disk and 3.6 GB in memory, so the repo reads it
as a **stream** of Arrow record batches: `conn.edge_batches()`.

In [ ]:
batches = conn.edge_batches()
pre, post, w = next(batches)
print("first batch:", len(pre), "rows")
print(pd.DataFrame({"body_pre": pre[:5], "body_post": post[:5], "weight": w[:5]}))
print()
print("synapses per connection in the FIRST batch:  median", int(np.median(w)), " max", int(w.max()))

The first batch is full of huge connections (median 125 synapses!). That is not a typo: **the file is sorted by weight, strongest first**. So the first batch says nothing about a typical connection.
To get a fair picture, sample batches evenly spaced across the file (this opens the same Arrow file `edge_batches` uses, reading only a few chunks).

In [ ]:
import pyarrow as pa

reader = pa.ipc.open_file(str(conn.paths["weights"]))
n_batches = reader.num_record_batches
picked = ((np.arange(60) + 0.5) * n_batches / 60).astype(int)       # 60 evenly spaced batches (the middle of each 1/60th of the file)
w = np.concatenate([reader.get_batch(int(i)).column("weight").to_numpy(zero_copy_only=False) for i in picked])
print(f"{n_batches} batches of 65,536 rows; looking at {len(picked)} of them ({len(w):,} connections)")
print("exactly 1 synapse  :", f"{(w == 1).mean():.0%}")
print("3 or fewer synapses:", f"{(w <= 3).mean():.0%}")
print(">= 5 synapses      :", f"{(w >= 5).mean():.0%}   <- the repo's N = 5 rule keeps only these")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.hist(np.clip(w, 0, 30), bins=30, color=H.TEAL)
ax.set_yscale("log"); ax.set_xlabel("synapses in a connection (clipped at 30)"); ax.set_ylabel("connections (log)")
ax.set_title("most connections are a single synapse")
plt.show()

Most connections are a single synapse; only a few percent have 5 or more. Notebook 02 uses the repo's **N = 5 rule** (keep connections of at least 5 synapses) when it adds neurons beyond the core cast.

## Recap

- `Connectome()` = annotations in memory + a streaming reader for the weights.
- A neuron = body id + type + side + soma + a sign from its transmitter.
- The repo picks neurons by **type**; counts are verified so the data can't silently change.

**Try this**
1. Find a type with a similar count to LC4 (126). What does it do? (Hint: look at `subclass`.)
2. Plot the soma positions of `Tm3` or `T3`. What shape do they make?
3. Count `DNa01` and `DNa02`: they're the steering neurons used by `groups.yaml`.

**Next:** notebook 02 turns these neurons and connections into the small dynamical system the fly runs on.